# _lib/rule_engine — Patroon A quarantine-routerings-hulpfuncties

Pure hulpfuncties voor het warn/drop/fail-ernstmodel dat in Silver gebruikt wordt.

Drop-regels voeden twee afgeleide hulpfuncties:
- `build_clean_predicate(rules)` → SQL-string om clean vs. quarantine te filteren.
- `build_failed_rules_expr(rules)` → Spark `Column` die een `ARRAY<STRING>`
  produceert van elke drop-regel die een rij schendt.

**Importeer via `%run ./_lib/rule_engine` vanuit het Silver DLT-pipeline-notebook.**

Herbruikbaar voor `order_detail` en toekomstige Silver-tabellen die dezelfde
Patroon A clean/quarantine-splitsing nodig hebben.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Column


def build_clean_predicate(drop_rules: dict) -> str:
    """AND-verbindt alle drop-regel-predicaten tot één SQL-string.

    Een rij is clean alleen als elk drop-predicaat naar TRUE evalueert. Lege drop_rules
    geeft '1=1' terug zodat de aanroeper het resultaat als filter kan toepassen
    zonder special-casing.
    """
    if not drop_rules:
        return "1=1"
    return " AND ".join(f"({predicate})" for predicate in drop_rules.values())


def build_failed_rules_expr(drop_rules: dict) -> Column:
    """Geeft een Column-expressie terug die een ARRAY<STRING> van gefaalde regelnamen produceert.

    Voor elk (name, predicate)-paar wordt een CASE WHEN NOT (predicate) THEN 'name'-
    kolom uitgegeven, daarna worden de namen verzameld in een array en NULLs
    eruit gefilterd via array_compact (Spark 3.4+ — DBR 13+).

    Geeft een lege array terug voor clean-rijen; in de praktijk roept alleen de
    quarantine-tak deze expressie aan.
    """
    rule_columns = [
        F.when(~F.expr(predicate), F.lit(name))
        for name, predicate in drop_rules.items()
    ]
    return F.array_compact(F.array(*rule_columns))


print("rule_engine geladen: build_clean_predicate, build_failed_rules_expr")